# 04 - Sensibilidad del costo administrativo en validación

Exploramos cómo cambia la decisión al variar Ca, manteniendo las mismas probabilidades y operaciones. Cada intervención cuesta Ca tanto si la transacción era fraude como si era legítima. Los valores 1, 5, 10, 20, 50 y 100 son escenarios ilustrativos en unidades del dataset; no son costos estimados de una entidad.

Trabajamos con la ventana posterior de validación del notebook 03. No reentrenamos XGBoost ni accedemos a test. Comparamos regla fija y BMR; el árbol requiere reentrenarse por Ca y se incorpora en la comparación parametrizada del notebook 03.

Antes de reportar ahorro debemos distinguir la referencia trivial del anteproyecto y la mejora frente al corte 0,5. Ninguna salida de este ejercicio es una conclusión económica final.


## 1. Recuperar las probabilidades

Comprobamos identificadores únicos, montos no negativos, etiquetas binarias y probabilidades válidas. La política fija debe corresponder exactamente al corte 0,5. Así atribuimos las diferencias de costo a las acciones y no a poblaciones distintas.


In [1]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from fraud_cost.scenarios import evaluate_hypothetical_scenarios, validate_validation_scores, HYPOTHETICAL_CA_GRID

PROJECT_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src").is_dir() and (p / "data").is_dir())
SCORES_PATH = PROJECT_ROOT / "data/interim/validation_scores.parquet"
if not SCORES_PATH.exists():
    raise FileNotFoundError("Ejecute notebook 03 para generar las probabilidades de validación")
scores = pd.read_parquet(SCORES_PATH)
amount, y, p, fixed = validate_validation_scores(scores)
print(f"Validación posterior: {len(scores):,} transacciones; {int(y.sum()):,} fraudes.")


Validación posterior: 42,047 transacciones; 1,517 fraudes.


## 2. Variar Ca y evaluar las acciones

BMR interviene cuando p_i × A_i supera Ca. Aprobar en empate evita intervenciones sin ventaja esperada. No dividimos por montos, por lo que el caso A_i=0 queda resuelto directamente.

Para cada escenario sumamos Ca por intervención y los montos de fraudes aprobados. Calculamos ahorro frente a min(costo de aprobar todo, costo de intervenir todo), y por separado la diferencia frente a la regla fija.


In [2]:
CA_GRID = HYPOTHETICAL_CA_GRID
scenario_results = evaluate_hypothetical_scenarios(scores, ca_grid=CA_GRID)
display(scenario_results[[
    "administrative_cost_ca", "strategy", "administrative_cost_total",
    "missed_fraud_amount", "cost_total_proxy", "reference_cost",
    "savings_fraction_vs_reference", "savings_fraction_vs_fixed_proxy",
    "fraud_recall", "legitimate_block_rate",
]])


,administrative_cost_ca,strategy,administrative_cost_total,missed_fraud_amount,cost_total_proxy,reference_cost,savings_fraction_vs_reference,savings_fraction_vs_fixed_proxy,fraud_recall,legitimate_block_rate
0,1.0,fixed_0_5,544.0,197856.601,198400.601,42047.000,-3.718544,0.000000,0.275544,0.003109
1,1.0,bmr,16322.0,9624.717,25946.717,42047.000,0.382912,0.869221,0.854318,0.370738
2,5.0,fixed_0_5,2720.0,197856.601,200576.601,210235.000,0.045941,0.000000,0.275544,0.003109
3,5.0,bmr,28730.0,38077.822,66807.822,210235.000,0.682223,0.666921,0.644034,0.117666
4,10.0,fixed_0_5,5440.0,197856.601,203296.601,254536.812,0.201308,0.000000,0.275544,0.003109
5,10.0,bmr,33390.0,55049.282,88439.282,254536.812,0.652548,0.564974,0.536585,0.062300
6,20.0,fixed_0_5,10880.0,197856.601,208736.601,254536.812,0.179936,0.000000,0.275544,0.003109
7,20.0,bmr,36000.0,81791.224,117791.224,254536.812,0.537233,0.435694,0.387607,0.029904
8,50.0,fixed_0_5,27200.0,197856.601,225056.601,254536.812,0.115819,0.000000,0.275544,0.003109
9,50.0,bmr,32500.0,130753.101,163253.101,254536.812,0.358627,0.274613,0.193804,0.008784


## 3. Exportar evidencia seleccionada

Guardamos la tabla completa y una figura de ahorro frente a la referencia trivial. Conservamos valores negativos: indican que una estrategia cuesta más que esa referencia. El eje representa Ca absoluto; no lo escalamos con la mediana de esta ventana.


In [3]:
TABLE_PATH = PROJECT_ROOT / "results/tables/escenarios_hipoteticos_validation.csv"
FIGURE_PATH = PROJECT_ROOT / "results/figures/ahorro_hipotetico_validation.png"
TABLE_PATH.parent.mkdir(parents=True, exist_ok=True)
FIGURE_PATH.parent.mkdir(parents=True, exist_ok=True)
scenario_results.to_csv(TABLE_PATH, index=False)
fig, ax = plt.subplots(figsize=(8, 4.8))
for strategy, block in scenario_results.groupby("strategy"):
    ax.plot(block.administrative_cost_ca, block.savings_fraction_vs_reference * 100, marker="o", label=strategy)
ax.axhline(0, color="gray", linestyle="--")
ax.set(xlabel="Ca hipotético (unidades del dataset)", ylabel="Ahorro frente a referencia trivial (%)",
       title="Sensibilidad de Ca en validación posterior")
ax.legend()
fig.tight_layout()
fig.savefig(FIGURE_PATH, dpi=160, bbox_inches="tight")
plt.show()


C:\Users\Milton Cuervo\AppData\Local\Temp\ipykernel_22872\1840849594.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4. Interpretar los resultados

Al aumentar Ca, BMR exige un riesgo de aprobación mayor y reduce las intervenciones. El recall de esa regla no aumenta al elevar Ca, pero el costo total no necesariamente es monótono: depende de los montos de fraude que se dejan pasar.

La política fija toma las mismas acciones para todos los escenarios. Su costo sí cambia porque cada intervención cuesta un Ca distinto. La referencia trivial también puede cambiar de aprobar todo a intervenir todo.

Una reducción del riesgo esperado no garantiza ahorro realizado positivo con probabilidades imperfectas. Leemos las cifras como sensibilidad de validación, con la unidad original del dataset y bajo el supuesto de que intervenir evita la pérdida fraudulenta. Los escenarios finales de Ca deben acordarse antes de evaluar test.
